In [1]:
# Install Edge TTS for audio generation and nest-asyncio for Jupyter compatibility
!pip install -q edge-tts nest-asyncio

In [10]:
import json
import os
import glob
import shutil
import asyncio
import edge_tts
import nest_asyncio
from datetime import datetime
from tqdm.notebook import tqdm

nest_asyncio.apply()

# Configuration
INPUT_DIR = "/content"
OUTPUT_DIR = "/content/dataset"
ZIP_OUTPUT = "/content/urdu_tts_dataset"
VOICE = "ur-PK-UzmaNeural"

# 1. Create directory if it doesn't exist (DO NOT WIPE IT)
os.makedirs(OUTPUT_DIR, exist_ok=True)

test_types = [
    "blocks", "prolongation", "repetitions", "velar_fronting",
    "stopping", "gliding", "cluster_reduction", "epenthesis"
]

# 2. Load existing manifest or initialize a new one
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")

if os.path.exists(manifest_path):
    print("Found existing manifest. Resuming previous session...")
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
else:
    print("No existing manifest found. Starting fresh...")
    manifest = {
        "version": "1.0.0",
        "generated_at": datetime.utcnow().isoformat(),
        "english": {tt: [] for tt in test_types},
        "urdu": {tt: [] for tt in test_types}
    }

def save_manifest():
    """Saves the manifest to disk immediately."""
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

No existing manifest found. Starting fresh...


/tmp/ipykernel_231/4098613982.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


In [11]:
# Speed configurations (0% is default, negative percentages slow it down)
SENTENCE_RATE = "+0%"
WORD_RATE = "-30%"  # Slows down single words by 30%

async def generate_audio_async(text, relative_path, speech_rate):
    """Generates audio only if it doesn't already exist, applying specific speed rates."""
    filepath = os.path.join(OUTPUT_DIR, relative_path)

    # RESUMABILITY: Skip if file already exists
    if os.path.exists(filepath):
        return "skipped"

    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    try:
        # Pass the rate parameter to Edge TTS
        communicate = edge_tts.Communicate(text, VOICE, rate=speech_rate)
        await communicate.save(filepath)
        return "generated"
    except Exception as e:
        print(f"\nError generating '{text}': {str(e)}")
        return "failed"

async def process_dataset():
    json_files = glob.glob(os.path.join(INPUT_DIR, "devanagari_urdu_*.json"))

    if not json_files:
        raise FileNotFoundError("Could not find the 8 JSON files in the root directory.")

    stats = {"generated": 0, "skipped": 0, "failed": 0}

    for file_path in sorted(json_files):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        test_type = data.get("test_type")
        sentences = data.get("sentences", [])

        if not test_type or test_type not in manifest["urdu"]:
            continue

        print(f"\nProcessing: {test_type}")

        for sentence_data in tqdm(sentences, leave=False):
            sent_id = sentence_data["id"]
            sent_text = sentence_data["text_original"]
            difficulty = sentence_data["difficulty"]

            sent_rel_path = f"urdu/{test_type}/sentence/{sent_id}.wav"

            # 1. Generate/Skip Sentence Audio (Normal Speed)
            res = await generate_audio_async(sent_text, sent_rel_path, SENTENCE_RATE)
            stats[res] += 1

            manifest_words = []
            targets = sentence_data.get("targets", [])

            # 2. Generate/Skip Word Audio (Slower Speed)
            for idx, target in enumerate(targets):
                word_urdu = target["word_original"]
                expected_ipa = target["expected_ipa"]

                word_rel_path = f"urdu/{test_type}/word/{sent_id}_w{idx}_{word_urdu}.wav"

                # Pass WORD_RATE here to slow down the isolated word
                w_res = await generate_audio_async(word_urdu, word_rel_path, WORD_RATE)
                stats[w_res] += 1

                manifest_words.append({
                    "index": idx,
                    "word": word_urdu,
                    "file": word_rel_path,
                    "expected_ipa": expected_ipa
                })

            # 3. Update Manifest in Memory
            new_sent_obj = {
                "id": sent_id,
                "text": sent_text,
                "difficulty": difficulty,
                "file": sent_rel_path,
                "words": manifest_words
            }

            existing_sents = manifest["urdu"][test_type]
            idx_to_replace = next((i for i, s in enumerate(existing_sents) if s["id"] == sent_id), None)

            if idx_to_replace is not None:
                existing_sents[idx_to_replace] = new_sent_obj
            else:
                existing_sents.append(new_sent_obj)

            # 4. Save manifest strictly after EVERY sentence finishes
            manifest["generated_at"] = datetime.utcnow().isoformat()
            save_manifest()

    return stats

# Run the async generation pipeline
stats = asyncio.run(process_dataset())
print(f"\nExecution Complete.")
print(f"Newly Generated : {stats['generated']} files")
print(f"Skipped (Cached): {stats['skipped']} files")
print(f"Failed          : {stats['failed']} files")


Processing: blocks


  0%|          | 0/45 [00:00<?, ?it/s]

/tmp/ipykernel_231/1424527703.py:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  manifest["generated_at"] = datetime.utcnow().isoformat()



Processing: prolongation


  0%|          | 0/45 [00:00<?, ?it/s]

...


Processing: cluster_reduction


  0%|          | 0/45 [00:00<?, ?it/s]


Processing: epenthesis


  0%|          | 0/45 [00:00<?, ?it/s]


Execution Complete.
Newly Generated : 1632 files
Skipped (Cached): 4 files
Failed          : 0 files


In [12]:
# Create ZIP archive of the entire dataset folder (audio + manifest)
print("Zipping dataset...")
shutil.make_archive(ZIP_OUTPUT, "zip", OUTPUT_DIR)
zip_mb = os.path.getsize(f"{ZIP_OUTPUT}.zip") / (1024 * 1024)

print(f"Done! Download your dataset at {ZIP_OUTPUT}.zip ({zip_mb:.1f} MB)")

Zipping dataset...
Done! Download your dataset at /content/urdu_tts_dataset.zip (9.8 MB)


In [13]:
import os
import json
import glob
import shutil
from datetime import datetime

# ==========================================
# CONFIGURATION
# ==========================================
ENGLISH_ZIP_PATH = "/content/english.zip"
INPUT_DIR = "/content" # Directory containing 1_blocks.json, 2_prolongation.json, etc.
OUTPUT_DIR = "/content/dataset"
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

# ==========================================
# 1. UNZIP ENGLISH AUDIO
# ==========================================
print("--- Step 1: Extracting English Audio ---")
if os.path.exists(ENGLISH_ZIP_PATH):
    shutil.unpack_archive(ENGLISH_ZIP_PATH, OUTPUT_DIR)
    print(f"Successfully extracted {ENGLISH_ZIP_PATH} into {OUTPUT_DIR}")
else:
    print(f"WARNING: {ENGLISH_ZIP_PATH} not found! Proceeding to scan existing files in {OUTPUT_DIR}/english if any.")

# ==========================================
# 2. UPDATE MANIFEST WITH CORRELATED DATA
# ==========================================
print("\n--- Step 2: Updating Unified Manifest ---")

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_PATH}. Please run the Urdu generation first.")

with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

# Grab the 8 English JSON files
english_json_files = glob.glob(os.path.join(INPUT_DIR, "[1-8]_*.json"))

if not english_json_files:
    print("WARNING: Could not find the English JSON files (e.g., 1_blocks.json) in /content.")

total_sent_expected = 0
total_sent_found = 0
total_word_expected = 0
total_word_found = 0

print(f"{'TEST TYPE':<20} | {'SENTENCES (Found/Total)':<25} | {'WORDS (Found/Total)':<25}")
print("-" * 75)

for file_path in sorted(english_json_files):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    test_type = data.get("test_type")
    sentences = data.get("sentences", [])

    if not test_type or test_type not in manifest["english"]:
        continue

    # Clear existing entries for this test type
    manifest["english"][test_type] = []

    # Category Counters
    cat_sent_expected = len(sentences)
    cat_sent_found = 0
    cat_word_expected = sum(len(s.get("targets", [])) for s in sentences)
    cat_word_found = 0

    for sentence_data in sentences:
        sent_id = sentence_data["id"]
        sent_text = sentence_data["text"]
        difficulty = sentence_data.get("difficulty", 1)

        sent_rel_path = f"english/{test_type}/sentence/{sent_id}.wav"
        sent_abs_path = os.path.join(OUTPUT_DIR, sent_rel_path)

        # Check if sentence audio exists
        if not os.path.exists(sent_abs_path):
            continue

        cat_sent_found += 1
        manifest_words = []
        targets = sentence_data.get("targets", [])

        for idx, target in enumerate(targets):
            word_eng = target["word"]
            expected_ipa = target["expected_ipa"]

            word_rel_path = f"english/{test_type}/word/{sent_id}_w{idx}_{word_eng}.wav"
            word_abs_path = os.path.join(OUTPUT_DIR, word_rel_path)

            # Check if word audio exists
            if os.path.exists(word_abs_path):
                cat_word_found += 1
                manifest_words.append({
                    "index": idx,
                    "word": word_eng,
                    "file": word_rel_path,
                    "expected_ipa": expected_ipa
                })

        # Build and append the validated sentence object
        manifest["english"][test_type].append({
            "id": sent_id,
            "text": sent_text,
            "difficulty": difficulty,
            "file": sent_rel_path,
            "words": manifest_words
        })

    # Formatting the log output for this specific test type
    sent_pct = (cat_sent_found / max(cat_sent_expected, 1)) * 100
    word_pct = (cat_word_found / max(cat_word_expected, 1)) * 100

    sent_log = f"{cat_sent_found}/{cat_sent_expected} ({sent_pct:.1f}%)"
    word_log = f"{cat_word_found}/{cat_word_expected} ({word_pct:.1f}%)"

    print(f"{test_type:<20} | {sent_log:<25} | {word_log:<25}")

    # Add to global totals
    total_sent_expected += cat_sent_expected
    total_sent_found += cat_sent_found
    total_word_expected += cat_word_expected
    total_word_found += cat_word_found

# Update the timestamp and save
manifest["generated_at"] = datetime.utcnow().isoformat()
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# ==========================================
# 3. SUMMARY LOGS
# ==========================================
print("-" * 75)
print("\n--- Step 3: Grand Totals ---")

sent_total_pct = (total_sent_found / max(total_sent_expected, 1)) * 100
word_total_pct = (total_word_found / max(total_word_expected, 1)) * 100

print(f"Total Sentences Indexed : {total_sent_found} / {total_sent_expected} ({sent_total_pct:.1f}%)")
print(f"Total Words Indexed     : {total_word_found} / {total_word_expected} ({word_total_pct:.1f}%)")
print(f"\nUnified Manifest successfully updated and saved at: {MANIFEST_PATH}")

--- Step 1: Extracting English Audio ---
Successfully extracted /content/english.zip into /content/dataset

--- Step 2: Updating Unified Manifest ---
TEST TYPE            | SENTENCES (Found/Total)   | WORDS (Found/Total)      
---------------------------------------------------------------------------
blocks               | 45/45 (100.0%)            | 156/156 (100.0%)         
prolongation         | 45/45 (100.0%)            | 168/168 (100.0%)         
repetitions          | 45/45 (100.0%)            | 168/169 (99.4%)          
velar_fronting       | 45/45 (100.0%)            | 135/135 (100.0%)         
stopping             | 45/45 (100.0%)            | 146/146 (100.0%)         
gliding              | 45/45 (100.0%)            | 136/136 (100.0%)         
cluster_reduction    | 45/45 (100.0%)            | 135/135 (100.0%)         
epenthesis           | 45/45 (100.0%)            | 141/145 (97.2%)          
---------------------------------------------------------------------------

--- 

/tmp/ipykernel_231/1955180093.py:127: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  manifest["generated_at"] = datetime.utcnow().isoformat()
